In [1]:
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.datasets import MovieLens100K
from sklearn.metrics import roc_auc_score

# 固定随机种子
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. 加载并处理MovieLens数据集
dataset = MovieLens100K(root='./data/MovieLens')
data = dataset[0]

# 构建用户-物品二部图
num_users = data['user'].num_nodes
num_movies = data['movie'].num_nodes
# 统一节点编号：用户0~num_users-1，电影num_users~num_users+num_movies-1
user_nodes = torch.arange(num_users)
movie_nodes = torch.arange(num_users, num_users + num_movies)

# 构建边索引：用户-电影交互边
edge_index = torch.stack([
    user_nodes[data['user', 'rates', 'movie'].edge_index[0]],
    movie_nodes[data['user', 'rates', 'movie'].edge_index[1]]
])
# 无向图，添加反向边
edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1).to(device)

# 构建节点特征：简单的one-hot特征，也可以用用户/电影的属性特征
x = torch.eye(num_users + num_movies).to(device)
num_nodes = x.shape[0]

# 构建训练/测试集：正样本（有交互）+ 负样本（无交互）
# 正样本：所有交互边
pos_edge_index = edge_index[:, :edge_index.shape[1]//2]
# 随机划分训练/测试集
num_edges = pos_edge_index.shape[1]
perm = torch.randperm(num_edges)
train_pos_edges = pos_edge_index[:, perm[:int(num_edges*0.8)]].to(device)
test_pos_edges = pos_edge_index[:, perm[int(num_edges*0.8):]].to(device)

# 生成负样本：随机采样无交互的用户-电影对
def generate_neg_edges(num_users, num_movies, pos_edges, num_neg):
    neg_edges = []
    pos_set = set((u.item(), m.item()) for u, m in pos_edges.T)
    while len(neg_edges) < num_neg:
        u = torch.randint(0, num_users, (1,)).item()
        m = torch.randint(num_users, num_users+num_movies, (1,)).item()
        if (u, m) not in pos_set and (m, u) not in pos_set:
            neg_edges.append([u, m])
    return torch.tensor(neg_edges).T.to(device)

train_neg_edges = generate_neg_edges(num_users, num_movies, pos_edge_index, train_pos_edges.shape[1])
test_neg_edges = generate_neg_edges(num_users, num_movies, pos_edge_index, test_pos_edges.shape[1])

Extracting data/MovieLens/ml-100k.zip
Processing...
Done!


In [2]:
# 2. 构建GCN推荐模型
class GNNRecommender(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        # 编码节点，生成用户/物品的嵌入
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.3, training=self.training)
        x = self.conv2(x, edge_index)
        return x

    def predict_link(self, z, edge_label_index):
        # 内积计算用户-物品的匹配度，值越高越推荐
        src = z[edge_label_index[0]]
        dst = z[edge_label_index[1]]
        return (src * dst).sum(dim=-1)

# 初始化模型
model = GNNRecommender(
    in_channels=num_nodes,
    hidden_channels=128,
    out_channels=64
).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [3]:
# 3. 训练模型
epochs = 100
for epoch in range(1, epochs+1):
    model.train()
    optimizer.zero_grad()
    # 生成节点嵌入
    z = model(x, edge_index)
    # 预测正/负样本
    pos_score = model.predict_link(z, train_pos_edges)
    neg_score = model.predict_link(z, train_neg_edges)
    # 二分类损失：BCE损失
    loss = F.binary_cross_entropy_with_logits(
        torch.cat([pos_score, neg_score]),
        torch.cat([torch.ones_like(pos_score), torch.zeros_like(neg_score)])
    )
    loss.backward()
    optimizer.step()

    # 验证
    if epoch % 10 == 0:
        model.eval()
        with torch.no_grad():
            z = model(x, edge_index)
            pos_score = model.predict_link(z, test_pos_edges)
            neg_score = model.predict_link(z, test_neg_edges)
            # 计算AUC指标，推荐系统核心指标
            y_true = torch.cat([torch.ones_like(pos_score), torch.zeros_like(neg_score)]).cpu().numpy()
            y_score = torch.cat([pos_score, neg_score]).sigmoid().cpu().numpy()
            auc = roc_auc_score(y_true, y_score)
            print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, 测试集AUC: {auc:.4f}')

Epoch: 010, Loss: 0.5862, 测试集AUC: 0.8533
Epoch: 020, Loss: 0.5507, 测试集AUC: 0.8393
Epoch: 030, Loss: 0.5228, 测试集AUC: 0.8491
Epoch: 040, Loss: 0.4910, 测试集AUC: 0.8649
Epoch: 050, Loss: 0.4477, 测试集AUC: 0.8867
Epoch: 060, Loss: 0.4479, 测试集AUC: 0.8799
Epoch: 070, Loss: 0.4068, 测试集AUC: 0.9017
Epoch: 080, Loss: 0.3951, 测试集AUC: 0.9063
Epoch: 090, Loss: 0.3861, 测试集AUC: 0.9070
Epoch: 100, Loss: 0.3794, 测试集AUC: 0.9113


In [8]:
# 4. 给指定用户生成推荐
def recommend_for_user(user_id, top_k=10):
    model.eval()
    with torch.no_grad():
        z = model(x, edge_index)
        # 该用户已交互的电影
        interacted_movies = pos_edge_index[1][pos_edge_index[0] == user_id].cpu().numpy()
        # 所有未交互的电影
        all_movies = movie_nodes.numpy()
        candidate_movies = torch.tensor([m for m in all_movies if m not in interacted_movies]).to(device)
        # 构建预测边
        predict_edges = torch.stack([
            torch.full_like(candidate_movies, user_id),
            candidate_movies
        ])
        # 预测匹配度
        scores = model.predict_link(z, predict_edges).sigmoid()
        # 取top_k推荐
        top_scores, top_indices = scores.topk(top_k)
        top_movie_ids = candidate_movies[top_indices].cpu().numpy() - num_users  # 转回原始电影ID
        return top_movie_ids, top_scores.cpu().numpy()

# 示例：给用户ID=0推荐10部电影
top_movies, top_scores = recommend_for_user(user_id=0, top_k=10)
print("\n给用户推荐的Top10电影ID与匹配度：")
for movie_id, score in zip(top_movies, top_scores):
    print(f"电影ID: {movie_id:4d}, 匹配度: {score:.4f}")


给用户推荐的Top10电影ID与匹配度：
电影ID:  173, 匹配度: 0.9876
电影ID:   99, 匹配度: 0.9850
电影ID:   97, 匹配度: 0.9845
电影ID:   55, 匹配度: 0.9843
电影ID:  120, 匹配度: 0.9838
电影ID:  209, 匹配度: 0.9793
电影ID:   68, 匹配度: 0.9768
电影ID:  116, 匹配度: 0.9745
电影ID:  221, 匹配度: 0.9719
电影ID:  422, 匹配度: 0.9703
